In [1]:
# Import the helpers used in this notebook.
# `json` reads the raw CUAD file and parses it into a Python dictionary.
import json

In [2]:
# Open the raw CUAD JSON file from disk and parse it into a Python dictionary.
with open(
    r"C:\Users\chari\Desktop\Contract_Intelligence_AI\data\raw\dataset\CUAD_v1\CUAD_v1.json",
    "r",
    encoding="utf-8"
) as f:
    data = json.load(f)

In [3]:
# Store the list of contract records in a shorter variable for exploration.
contracts = data["data"]

In [4]:
import json
import random
import pandas as pd
from collections import Counter

# ==========================================
# LOAD CUAD DATASET
# ==========================================

with open(
    r"C:\Users\chari\Desktop\Contract_Intelligence_AI\data\raw\dataset\CUAD_v1\CUAD_v1.json",
    "r",
    encoding="utf-8"
) as f:
    data = json.load(f)

contracts = data["data"]

print("Dataset Loaded Successfully")
print("Total Contracts:", len(contracts))


# ==========================================
# TARGET LABELS
# ==========================================

target_labels = [
    "Termination For Convenience",
    "Renewal Term",
    "Cap On Liability",
    "Uncapped Liability"
]

print("\nTarget Labels:")
for label in target_labels:
    print("-", label)


# ==========================================
# HELPER: EXTRACT CLAUSE-LEVEL TEXT
# ==========================================

CONTEXT_WINDOW = 250   # chars of surrounding context on each side
NEGATIVE_SNIPPET_LEN = 500  # target length for negative examples

random.seed(42)  # reproducibility


def extract_positive_text(context, answers):
    """Extract the answer span with surrounding context for a POSITIVE example."""
    # Use the longest answer span for best coverage
    best = max(answers, key=lambda a: len(a["text"]))
    start = best["answer_start"]
    end = start + len(best["text"])

    # Expand window to include surrounding context
    window_start = max(0, start - CONTEXT_WINDOW)
    window_end = min(len(context), end + CONTEXT_WINDOW)

    snippet = context[window_start:window_end].strip()
    # Clean up excessive whitespace
    snippet = " ".join(snippet.split())
    return snippet


def extract_negative_text(context):
    """Extract a random snippet from the contract for a NEGATIVE example."""
    cleaned = " ".join(context.split())
    if len(cleaned) <= NEGATIVE_SNIPPET_LEN:
        return cleaned
    start = random.randint(0, len(cleaned) - NEGATIVE_SNIPPET_LEN)
    snippet = cleaned[start:start + NEGATIVE_SNIPPET_LEN].strip()
    return snippet


# ==========================================
# CREATE TRAINING DATA
# ==========================================

texts = []
labels = []

for contract in contracts:

    for paragraph in contract["paragraphs"]:

        context = paragraph["context"]

        for qa in paragraph["qas"]:

            label = qa["id"].split("__")[-1]

            # Only keep selected labels
            if label in target_labels:

                if not qa["is_impossible"] and qa["answers"]:
                    # POSITIVE: extract the specific clause with context
                    snippet = extract_positive_text(context, qa["answers"])
                    target = 1
                else:
                    # NEGATIVE: extract a random excerpt from the contract
                    snippet = extract_negative_text(context)
                    target = 0

                texts.append(snippet)
                labels.append({
                    "label_name": label,
                    "target": target
                })


# ==========================================
# CREATE DATAFRAME
# ==========================================

df = pd.DataFrame({
    "text": texts,
    "label_name": [x["label_name"] for x in labels],
    "target": [x["target"] for x in labels]
})

print("\nDataset Shape:")
print(df.shape)

# Show text length stats
print("\nText Length Stats (characters):")
print(df["text"].str.len().describe())


# ==========================================
# CHECK CLASS DISTRIBUTION
# ==========================================

print("\nClass Distribution:\n")

for label in target_labels:

    subset = df[df["label_name"] == label]

    counts = Counter(subset["target"])

    print(label)
    print(f"Present   (1): {counts[1]}")
    print(f"Absent    (0): {counts[0]}")
    print("-" * 40)


# ==========================================
# SHOW SAMPLE DATA
# ==========================================

print("\nSample Dataset Rows:\n")

pd.set_option("display.max_colwidth", 120)
display(df.head(10))
pd.reset_option("display.max_colwidth")


# ==========================================
# SAVE DATASET
# ==========================================

output_path = r"C:\Users\chari\Desktop\Contract_Intelligence_AI\data\processed\clause_classification_dataset.csv"

df.to_csv(output_path, index=False)

print("\nProcessed dataset saved successfully!")
print("Saved at:")
print(output_path)

Dataset Loaded Successfully
Total Contracts: 510

Target Labels:
- Termination For Convenience
- Renewal Term
- Cap On Liability
- Uncapped Liability

Dataset Shape:
(2040, 3)

Text Length Stats (characters):
count    2040.000000
mean      637.945588
std       253.577964
min       461.000000
25%       500.000000
50%       500.000000
75%       718.000000
max      3253.000000
Name: text, dtype: float64

Class Distribution:

Termination For Convenience
Present   (1): 183
Absent    (0): 327
----------------------------------------
Renewal Term
Present   (1): 176
Absent    (0): 334
----------------------------------------
Cap On Liability
Present   (1): 275
Absent    (0): 235
----------------------------------------
Uncapped Liability
Present   (1): 111
Absent    (0): 399
----------------------------------------

Sample Dataset Rows:



,text,label_name,target
0,"nt shall be ten (10) years (the ""Term"") which shall commence on the date upon which the Company delivers to Distribu...",Renewal Term,1
1,"wo, the expectation for year two will have been met, but there will be no carry-over to year three. If the Distribut...",Termination For Convenience,0
2,"Appointment. The Company appoints the Distributor as an exclusive distributor of Products in the Market, subject to ...",Uncapped Liability,0
3,"nder shall be free from defects in design, materials and workmanship for a period of twenty-four (24) months after d...",Cap On Liability,0
4,"ime the relevant liability is to be assessed (the ""Applicable Time""), it shall be calculated on a pro-rata basis as ...",Renewal Term,0
5,ve Google a reasonable time to fix the problem and (if necessary) to supply Distributor with a corrected or replacem...,Termination For Convenience,0
6,"'s liability under Clause 10 (Indemnities), or Distributor's liability under Clause 2 (License Grants and Restrictio...",Uncapped Liability,1
7,"'s liability under Clause 10 (Indemnities), or Distributor's liability under Clause 2 (License Grants and Restrictio...",Cap On Liability,1
8,"is commissioned the buyer. 5. PACKING: To be packed in new strong wooden case(s) /carton(s), suitable for long dista...",Renewal Term,0
9,"s forward amendments or not accept orders, the seller shall be in the form of a written notice to entrusted party, e...",Termination For Convenience,0



Processed dataset saved successfully!
Saved at:
C:\Users\chari\Desktop\Contract_Intelligence_AI\data\processed\clause_classification_dataset.csv
